# Deliverable 3 - INF201 

Submission deadline: 13th Nov 2026

### Hand-in information:
* This is a group submission, so one person submits on behalf of the pair/group
* Submit Python scripts (.py), a Jupyter notebook (ipynb) <i>and</i> a pdf of the same notebook
    * See Overview and objectives for a list of what your notebook should contain
* Include a link to your GitHub repository and a log of the commits
    * To generate the log, paste this into the terminal after your last commit:<br>
    `git log --graph --pretty=format:"%h - %an, %ar : %s" > git_log.txt`
    * The repository should contain a `requirements.txt` with information on your virtual environment
    * Note: Remember to create a new virtual environment for each deliverable
* Include a discussion of GenAI use, for example a summary of the chat log as you have
done earlier
    * This can be a separate file or be included in the Jupyter notebook
* Make a short (5 mins) video discussing your code and results
    * Note that you both have to contribute to the video
    * Either by recording it together
    * Or recording parts separately and editing it together
* We would appriciate it if use English for this assignment, both in the writing and in the video

## Overview and objectives

In this assignment you will continue working on the Character class in `game_simulator_v1.py` that we created in lecture_08 (check out the Week 44 module), and expand this to create a mini-RPG in the form of a battle simulator. To do this you will update and extend the existing class and methods, introduce new classes allowing us to use dice and weapons, and create a combat manager class.

The rules for the game is as follows:
* The game you build should allow two characters to battle until one of them wins. 
* The characters can use weapons to attack, where the damage dealt is determined by a dice roll.
* The characters can have different specialisations (warrior or mage). 
* A character's HP (hit points, or health) must be between 0 and the max given for this specific character.
* The character dies when it reaches 0 HP.

Learning Objectives:
* Enforce data encapsulation using private attributes and properties.
* Implement custom classes to model discrete concepts (`Dice`, `Weapon`, `Battle`).¨
* Create subclasses inheriting from the `Character` class.
* Apply object composition by equipping characters with weapons and dice.

To organise your project, we recommend the following struckture:
* Separate modules (Python scripts) for each of the classes created
* A Jupyter Notebook containing the following:
    * A presentation of each class created (showing attributes and methods objects of the class has)
    * Simulated battles between different types of characters, with and without weapons

## Task 1: Update the Character class

When we created the Character class in the lecture we first included an `attack()` method, and afterwards indicated that we wanted hit points to be a protected feature `_hp`. This lead to the `Character.attack()` method changing the target's protected attribute. Modifying another object's private attribute is bad form! Your first task is to fix this.

* Implement a `take_damage()` method to be used when the `Character` object is injured. This method should only affect the characters *own* HP.
* Implement a `heal()` method to be used when the `Character` heals (in other words increase their hit points)
* Change the existing `attack()` method so that calls the target's `take_damage()` method rather than directly modifying the target's `_hp`.
* Add an `is_alive` property to the `Character` class. This should be read-only, and return True if `_hp`$ > 0$ and False otherwise.

In [1]:
# Setup: import all our classes from the separate modules
import random

from dice import Dice
from items import Weapon
from character import Character, Warrior, Mage
from battle import Battle

# Fixed seed so the "random" results are the same every time we run the notebook
random.seed(201)

Knight with sword (1-8):
2
2
1
6
1
Peasant without weapon (always 10):
10
10
10
58
5
95
95
100
True
30
Turn 1: dealt 20, mana left 20
Turn 2: dealt 20, mana left 10
Turn 3: dealt 20, mana left 0
Turn 4: dealt 5, mana left 5
Turn 5: dealt 3, mana left 10
Turn 6: dealt 20, mana left 0


### Solution: Updated `Character` class

The `Character` class (in `character.py`) now respects encapsulation:

| Member | Type | Description |
|---|---|---|
| `name`, `hp_max`, `damage`, `weapon` | attributes | Basic character data |
| `_hp` | private attribute | Current hit points, only changed by the character itself |
| `hp` | read-only property | Inspect current HP |
| `is_alive` | read-only property | `True` if `_hp > 0` |
| `take_damage(amount)` | method | Reduce own HP (never below 0) |
| `heal(amount)` | method | Increase own HP (never above `hp_max`) |
| `attack(target)` | method | Calls `target.take_damage(...)` instead of modifying `target._hp` |

Both `take_damage()` and `heal()` return the amount actually applied, and raise a `ValueError` for negative amounts.

In [2]:
alice = Character("Alice")
bob = Character("Bob")
print(alice)
print(repr(bob))

# Attack uses bob.take_damage() internally
alice.attack(bob)
print(f"Bob after one attack: {bob.hp} HP")

# HP cannot exceed hp_max ...
restored = bob.heal(500)
print(f"Bob healed {restored} HP and now has {bob.hp} HP")

# ... and cannot drop below 0
bob.take_damage(999)
print(f"Bob after 999 damage: {bob.hp} HP, alive: {bob.is_alive}")

# hp and is_alive are read-only
try:
    bob.hp = 1000
except AttributeError as error:
    print(f"Cannot set hp directly: {error}")

Character: Alice with 100 HP and 10 damage
<Character name=Bob hp=100 damage=10>
Bob after one attack: 90 HP
Bob healed 10 HP and now has 100 HP
Bob after 999 damage: 0 HP, alive: False
Cannot set hp directly: property 'hp' of 'Character' object has no setter


## Task 2: Introducing randomness with a Dice class

Currently, characters deal fixed damage. To introduce variable damage, create a separate `Dice` class.

* Create a `Dice` class that can create any n-sided Dice (n>=2), where the default number of sides is 6.
* Add a `roll()` method to roll the dice, using `random.randint(1, self.sides)`.
* Add a `__repr__` method that returns string representations like `Dice(sides=6)`.
* Implement a `__call__` method that allow you to roll the dice created by `d20 = Dice(sides=20)` by calling `d20.roll()`.
    * See the [Python documentation of `__call__`](https://docs.python.org/3/reference/datamodel.html#object.__call__) or [this example](https://www.w3schools.com/python/python_magic_call.asp)

### Solution: `Dice` class

The `Dice` class (in `dice.py`) models an n-sided dice:

| Member | Type | Description |
|---|---|---|
| `_sides` / `sides` | private attribute / read-only property | Number of sides (default 6, must be at least 2) |
| `roll()` | method | Returns `random.randint(1, sides)` |
| `__repr__()` | magic method | Returns e.g. `Dice(sides=6)` |
| `__call__()` | magic method | Makes the object callable, so `d20()` does the same as `d20.roll()` |

Note: the task text says `__call__` should let us roll by calling `d20.roll()`, but `roll()` works without `__call__`. What `__call__` actually enables is calling the object itself: `d20()`.

In [3]:
from collections import Counter

d6 = Dice()
d20 = Dice(sides=20)
print(d6, d20)

print("d6.roll():", d6.roll())
print("d20.roll():", d20.roll())
print("d20():", d20())  # __call__

# Roll a d6 many times to check that all sides come up about equally often
counts = Counter(d6() for _ in range(6000))
print("6000 rolls of d6:", dict(sorted(counts.items())))

# A dice needs at least 2 sides
try:
    Dice(sides=1)
except ValueError as error:
    print(f"Invalid dice: {error}")

Dice(sides=6) Dice(sides=20)
d6.roll(): 1
d20.roll(): 11
d20(): 14
6000 rolls of d6: {1: 1032, 2: 982, 3: 953, 4: 1052, 5: 1001, 6: 980}
Invalid dice: A dice must have at least 2 sides


## Task 3: Create items

Create a `Weapon` class in `items.py` that utilizes your `Dice` class for damage calculations. We call the module `items.py` rather than `weapons.py` so that it can be expanded to include other items your characters can hold.

* Define the class `Weapon` with the attributes `name` and `damage_dice: Dice`.
* Include a `roll_damage()` method that calls `self.damage_dice.roll()`
* Update `Character.__init__` to accept an optional weapon.
* If a character has an equipped weapon, `attack()` should roll damage from the weapon instead of using default base damage.

### Solution: `Weapon` class and armed characters

The `Weapon` class (in `items.py`) is an example of **composition**: a weapon *has a* `Dice`, and a character *has a* (optional) `Weapon`.

| Member | Type | Description |
|---|---|---|
| `name` | attribute | Name of the weapon |
| `damage_dice` | attribute (`Dice`) | The dice that determines damage |
| `roll_damage()` | method | Delegates to `self.damage_dice.roll()` |
| `__repr__()` | magic method | Returns e.g. `Weapon(name='Sword', damage_dice=Dice(sides=8))` |

`Character.__init__` now takes an optional `weapon` (default `None`). In `attack()`, the character rolls the weapon's dice if it has one, and otherwise uses its fixed base `damage`.

In [4]:
sword = Weapon("Sword", Dice(sides=8))
dagger = Weapon("Dagger", Dice(sides=4))
print(sword)
print(dagger)
print("Sword damage roll:", sword.roll_damage())

knight = Character("Knight", weapon=sword)
peasant = Character("Peasant")  # no weapon
dummy = Character("Dummy", hp_max=1000)

knight_hits = [knight.attack(dummy) for _ in range(5)]
peasant_hits = [peasant.attack(dummy) for _ in range(5)]
print("Knight with sword (d8):", knight_hits)
print("Peasant without weapon:", peasant_hits)

Weapon(name='Sword', damage_dice)=Dice(sides=8))
Weapon(name='Dagger', damage_dice)=Dice(sides=4))
Sword damage roll: 6
Knight with sword (d8): [2, 4, 6, 5, 7]
Peasant without weapon: [10, 10, 10, 10, 10]


## Task 4: Character specialization

Extend the character module to include two specialised subclasses of `Character`.

1. `Warrior` subclass (defensive specialist)
    * New attribute: `armor: int`, with a default value of 5
    * Override the `Character.take_damage()` method by reducing the incoming damage by armor before applying it.
2. `Mage` subclass (resource-based attacker)
    * New attributes: `mana: int` and `mana_max: int`, both with default value 30
    * Magic attacks consume mana, so the mana available determines whether the mage can use magic.
    * Override the `Character.attack()` method:
        * If the mage has at least 10 mana left, spend 10 mana to deal a magical spell attack dealing double damage. This magic attack does not use any weapons, even if the character has one equipped.
        * If the mage has less than 10 mana left, fall back to the base attack, and restore 5 mana. This base attack should use whichever weapon the character has equipped.

### Solution: `Warrior` and `Mage` subclasses

Both classes **inherit** from `Character` (an *is-a* relationship), so they get `hp`, `is_alive`, `heal()` etc. for free and only override what is different. Both use `super()` to reuse the parent's logic instead of copying it.

**`Warrior(Character)`** (defensive specialist)

| Member | Type | Description |
|---|---|---|
| `armor` | attribute | Default 5 |
| `take_damage(amount)` | overridden method | Reduces incoming damage by `armor` (never below 0), then calls `super().take_damage()` |

**`Mage(Character)`** (resource-based attacker)

| Member | Type | Description |
|---|---|---|
| `_mana` / `mana` | private attribute / read-only property | Default 30 |
| `mana_max` | attribute | Default 30 |
| `attack(target)` | overridden method | With at least 10 mana: spend 10 and cast a spell for 2 × base damage (ignores weapon). Otherwise: normal attack via `super().attack()` (uses weapon), then regain 5 mana |

Because `Character.attack()` calls `target.take_damage()`, a Warrior's armor works automatically no matter who attacks it. This is **polymorphism**, and it is the reason Task 1 was important.

In [5]:
# --- Warrior: armor reduces incoming damage ---
tank = Warrior("Tank")
print(f"{tank.name} has {tank.armor} armor")

dealt = peasant.attack(tank)  # 10 damage - 5 armor
print(f"Peasant hits Tank: {dealt} damage taken, Tank has {tank.hp} HP")

weak = Character("Weak", damage=3)
dealt = weak.attack(tank)  # 3 - 5 -> 0, armor cannot heal
print(f"Weak hits Tank: {dealt} damage taken, Tank has {tank.hp} HP")

tank.heal(5)  # inherited from Character
print(f"Tank after heal (inherited method): {tank.hp} HP")

# --- Mage: spells cost mana, normal attacks restore mana ---
merlin = Mage("Merlin", weapon=Weapon("Staff", Dice(sides=6)))
target = Character("Target", hp_max=1000)
print(f"\n{merlin.name} starts with {merlin.mana} mana")

for turn in range(1, 7):
    dealt = merlin.attack(target)
    print(f"Turn {turn}: dealt {dealt:2d} damage, mana left {merlin.mana:2d}")

Tank has 5 armor
Peasant hits Tank: 5 damage taken, Tank has 95 HP
Weak hits Tank: 0 damage taken, Tank has 95 HP
Tank after heal (inherited method): 100 HP

Merlin starts with 30 mana
Turn 1: dealt 20 damage, mana left 20
Turn 2: dealt 20 damage, mana left 10
Turn 3: dealt 20 damage, mana left  0
Turn 4: dealt  5 damage, mana left  5
Turn 5: dealt  1 damage, mana left 10
Turn 6: dealt 20 damage, mana left  0


## Task 5: Running battles

Create a `Battle` class to manage turn-based combat between two `Character` objects.

* Initialize `Battle` with `hero: Character` and `enemy: Character`.
* Implement a `play_round()` method where `hero` attacks `enemy`, and if `enemy` is still alive, `enemy` attacks `hero`.
* Implement a `simulate_battle()` method with a `while` loop that runs rounds until one character is no longer alive.

### Solution: `Battle` class

The `Battle` class (in `battle.py`) manages turn-based combat between two characters:

| Member | Type | Description |
|---|---|---|
| `hero`, `enemy` | attributes (`Character`) | The two fighters |
| `round` | attribute | Number of rounds played so far |
| `play_round()` | method | Hero attacks enemy; if the enemy survives, it attacks back |
| `simulate_battle(max_rounds=1000)` | method | Plays rounds in a `while` loop until one character dies. Returns the winner, or `None` for a draw |

`Battle` never checks *what kind* of character it is dealing with; it only calls `attack()`, and each subclass handles the details itself.

**Extra safety limit:** if neither fighter can damage the other (e.g. two warriors whose armor is higher than the incoming damage), the `while` loop would never end. We therefore added `max_rounds`. After the loop we check *why* it stopped: if both are still alive, the battle is a draw.

**Note:** characters keep their HP after a battle, so we create new characters for every battle below.

In [6]:
print("=== Battle 1: Character vs Character (no weapons) ===")
Battle(Character("Hero"), Character("Villain")).simulate_battle()

print("\n=== Battle 2: Warrior vs Mage (no weapons) ===")
Battle(Warrior("Conan"), Mage("Merlin")).simulate_battle()

=== Battle 1: Character vs Character (no weapons) ===
Round 1: Hero hits Villain for 10 damage (Villain: 90 HP left.
Villain hits Hero for 10 damage (Hero: 90 HP left.)
Round 2: Hero hits Villain for 10 damage (Villain: 80 HP left.
Villain hits Hero for 10 damage (Hero: 80 HP left.)
Round 3: Hero hits Villain for 10 damage (Villain: 70 HP left.
Villain hits Hero for 10 damage (Hero: 70 HP left.)
Round 4: Hero hits Villain for 10 damage (Villain: 60 HP left.
Villain hits Hero for 10 damage (Hero: 60 HP left.)
Round 5: Hero hits Villain for 10 damage (Villain: 50 HP left.
Villain hits Hero for 10 damage (Hero: 50 HP left.)
Round 6: Hero hits Villain for 10 damage (Villain: 40 HP left.
Villain hits Hero for 10 damage (Hero: 40 HP left.)
Round 7: Hero hits Villain for 10 damage (Villain: 30 HP left.
Villain hits Hero for 10 damage (Hero: 30 HP left.)
Round 8: Hero hits Villain for 10 damage (Villain: 20 HP left.
Villain hits Hero for 10 damage (Hero: 20 HP left.)
Round 9: Hero hits Villain

<Character name=Conan hp=5 damage=10>

In [7]:
print("=== Battle 3: Warrior with sword vs Mage with staff ===")
conan = Warrior("Conan", weapon=Weapon("Sword", Dice(sides=8)))
merlin = Mage("Merlin", weapon=Weapon("Staff", Dice(sides=6)))
Battle(conan, merlin).simulate_battle()

print("\n=== Battle 4: Mage with staff vs Character with dagger ===")
morgana = Mage("Morgana", weapon=Weapon("Staff", Dice(sides=6)))
thief = Character("Thief", weapon=Weapon("Dagger", Dice(sides=4)))
Battle(morgana, thief).simulate_battle()

=== Battle 3: Warrior with sword vs Mage with staff ===
Round 1: Conan hits Merlin for 4 damage (Merlin: 96 HP left.
Merlin hits Conan for 15 damage (Conan: 85 HP left.)
Round 2: Conan hits Merlin for 3 damage (Merlin: 93 HP left.
Merlin hits Conan for 15 damage (Conan: 70 HP left.)
Round 3: Conan hits Merlin for 2 damage (Merlin: 91 HP left.
Merlin hits Conan for 15 damage (Conan: 55 HP left.)
Round 4: Conan hits Merlin for 1 damage (Merlin: 90 HP left.
Merlin hits Conan for 0 damage (Conan: 55 HP left.)
Round 5: Conan hits Merlin for 1 damage (Merlin: 89 HP left.
Merlin hits Conan for 0 damage (Conan: 55 HP left.)
Round 6: Conan hits Merlin for 8 damage (Merlin: 81 HP left.
Merlin hits Conan for 15 damage (Conan: 40 HP left.)
Round 7: Conan hits Merlin for 4 damage (Merlin: 77 HP left.
Merlin hits Conan for 0 damage (Conan: 40 HP left.)
Round 8: Conan hits Merlin for 3 damage (Merlin: 74 HP left.
Merlin hits Conan for 0 damage (Conan: 40 HP left.)
Round 9: Conan hits Merlin for 1 dam

<Character name=Morgana hp=77 damage=10>

In [8]:
print("=== Battle 5: Two heavily armored warriors (draw) ===")
Battle(Warrior("Tank 1", armor=20), Warrior("Tank 2", armor=20)).simulate_battle(max_rounds=3)

=== Battle 5: Two heavily armored warriors (draw) ===
Round 1: Tank 1 hits Tank 2 for 0 damage (Tank 2: 100 HP left.
Tank 2 hits Tank 1 for 0 damage (Tank 1: 100 HP left.)
Round 2: Tank 1 hits Tank 2 for 0 damage (Tank 2: 100 HP left.
Tank 2 hits Tank 1 for 0 damage (Tank 1: 100 HP left.)
Round 3: Tank 1 hits Tank 2 for 0 damage (Tank 2: 100 HP left.
Tank 2 hits Tank 1 for 0 damage (Tank 1: 100 HP left.)
draw! NOBODY won after 3 rounds


## Conclusion

In this assignment we built a small battle game with four classes in separate files:
`Character` (with `Warrior` and `Mage`), `Dice`, `Weapon` and `Battle`.

### What we learned

- **Encapsulation:** HP is private (`_hp`). Other objects cannot change it directly. A character can only lose or gain HP through its own `take_damage()` and `heal()` methods.
- **Composition:** A weapon *has* a dice, and a character *has* a weapon. Each class does one small job.
- **Inheritance:** `Warrior` and `Mage` are special kinds of `Character`. They reuse the parent's code with `super()` and only change what is different.
- **Polymorphism:** `attack()` does not need to know if the target is a Warrior. The Warrior's own `take_damage()` method applies the armor automatically.

### What we saw in the battles

- The character who attacks first has an advantage. In Battle 1, the hero wins because the villain dies before it can hit back.
- The Mage is strong at the start, but becomes weaker when it runs out of mana.
- Weapons make the results random. We used `random.seed()` so the results are the same every time we run the notebook.
- Two warriors with very high armor can never hurt each other. We added a `max_rounds` limit so the battle ends in a draw instead of running forever.

### What could be improved

- Roll a dice to decide who attacks first, so the battle is fair.
- Add more items, for example healing potions.